In [1]:
import pandas as pd
import os
import numpy as np

import sys
sys.path.append('..')
from helpers import get_county2zone

In [2]:
def get_baseline_load_profiles():
    baseline_load_profiles = {}
    hourly_demand_path = "../data/baseline_load_profiles/processed"
    for fname in os.listdir(hourly_demand_path):
        fpath = os.path.join(hourly_demand_path, fname)
        if os.path.isdir(fpath):
            continue
        df = pd.read_csv(fpath, parse_dates=["timestamp"], index_col="timestamp")
        eia_code = fname.replace('.csv', '')
        baseline_load_profiles[eia_code] = df

    # Convert all profiles to Central time to match other ReEDS time series data
    baseline_load_profiles_cst = {}
    for eia_code, df in baseline_load_profiles.items():
        df = df.apply(lambda x: np.roll(x, shift=-6))
        df = df.loc[df.index.year <= 2023]
        df = pd.concat([df.loc[v] for g, v in df.groupby(df.index.year).groups.items()])
        baseline_load_profiles_cst[eia_code] = df.set_index(df.index.tz_localize('Etc/GMT+6'))

    return baseline_load_profiles_cst

In [3]:
# Create name mappings for EIA-930 respondents, subregions, etc.
hourly_rto_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_rto.csv"
)

hourly_subregion_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_subregion.csv",
    dtype={'subba': str}
)

def get_subbas(ba):
    subbas = hourly_subregion_demand.loc[hourly_subregion_demand.parent == ba].subba.unique().tolist()
    return subbas

In [4]:
def get_rooftop_pv_cf_profile(sector):
    df_cf = pd.read_hdf(f'../data/distpv_profiles/processed/county_rooftop_pv_cf_{sector}.h5')
    df_cf = df_cf.loc[df_cf.index.year.isin(range(2016, 2024))]

    return df_cf

In [5]:
def rescale_profile(profile, annual_totals):
    profile_norm = profile.apply(lambda x: x / x.groupby(x.index.year).transform('sum'))
    profile_norm = profile_norm.set_index(profile_norm.index.year, append=True)
    rescaled_profile = (
        profile_norm.mul(annual_totals.T, level=1)
        .fillna(0)
        .droplevel(1)
    )

    return rescaled_profile

In [6]:
baseline_load_profiles_cst = get_baseline_load_profiles()
distpv_residential_cf_profiles = get_rooftop_pv_cf_profile('residential')
distpv_commercial_cf_profiles = get_rooftop_pv_cf_profile('commercial')

In [8]:
retail_sales = pd.read_csv(
    '../data/county_ftm_sales_estimates.csv',
    index_col=['FIPS']
)
retail_sales.columns = [int(col) for col in retail_sales.columns]

direct_use = pd.read_csv('../data/county_direct_use.csv', index_col=['FIPS', 'is_pv'])
direct_use.columns = [int(col) for col in direct_use.columns]
direct_use = direct_use.groupby(direct_use.index.get_level_values('FIPS')).sum()

distpv_residential_consumption = pd.read_csv(
    '../data/county_distpv_residential_consumption.csv',
    index_col=['FIPS']
)
distpv_residential_consumption.columns = [int(col) for col in distpv_residential_consumption.columns]

distpv_non_residential_consumption = pd.read_csv(
    '../data/county_distpv_non_residential_consumption.csv',
    index_col=['FIPS']
)
distpv_non_residential_consumption.columns = [int(col) for col in distpv_non_residential_consumption.columns]

county_zone_area_coverage = (
    pd.read_csv('../data/county_zone_area_coverage.csv')
    .rename(columns={'rb': 'FIPS', 'coverage': 'zonal_coverage'})
    .set_index(['FIPS', 'EIAcode'])
)

In [9]:
retail_sales_and_direct_use = (
    retail_sales.add(direct_use, fill_value=0)
    .merge(
        county_zone_area_coverage,
        left_index=True,
        right_index=True
    )
)

for year in range(2016, 2024):
    retail_sales_and_direct_use[year] *= retail_sales_and_direct_use['zonal_coverage']

retail_sales_and_direct_use = retail_sales_and_direct_use.drop(columns='zonal_coverage')

zone_load_profiles = pd.concat({k:v['value'] for k,v in baseline_load_profiles_cst.items()}, axis=1)

In [10]:
rsdu_profiles_by_county = {}
for county in retail_sales_and_direct_use.index.get_level_values('FIPS').unique():
    county_rsdu_by_zone = retail_sales_and_direct_use.loc[county]
    zones = list(county_rsdu_by_zone.index)
    county_zone_load_profiles = zone_load_profiles[zones].copy()
    county_rsdu_profile = rescale_profile(
        county_zone_load_profiles, county_rsdu_by_zone
    )
    rsdu_profiles_by_county[county] = county_rsdu_profile.sum(axis=1)

county_rsdu_profiles = pd.concat(rsdu_profiles_by_county, axis=1)
county_distpv_residential_consumption_profiles = rescale_profile(
    distpv_residential_cf_profiles, distpv_residential_consumption
)
county_distpv_non_residential_consumption_profiles = rescale_profile(
    distpv_commercial_cf_profiles, distpv_non_residential_consumption
)

In [11]:
county_load_profiles = (
    county_rsdu_profiles
    .add(county_distpv_residential_consumption_profiles)
    .add(county_distpv_non_residential_consumption_profiles)
)

In [12]:
os.makedirs('../data/outputs', exist_ok=True)
county_load_profiles.to_hdf(
    '../data/outputs/historic_load_hourly_2016_2023_county.h5',
    key='data'
)

In [14]:
import sys
sys.path.append('..')
from helpers import get_county2zone

county2zone = get_county2zone(2023)

In [46]:
county_ba_map = dict(zip(county2zone['FIPS'], county2zone['ba']))

ba_load_profiles = county_load_profiles.copy()
ba_load_profiles.columns = ba_load_profiles.columns.map(county_ba_map)
ba_load_profiles = ba_load_profiles.T.groupby(ba_load_profiles.columns).sum().T

In [47]:
ba_load_profiles.to_hdf(
    '../data/outputs/historic_load_hourly_2016_2023_ba.h5',
    key='data'
)